# 01 — LFM2.5 Text: QLoRA-Finetuning auf Colab

Trainiert einen LoRA-Adapter auf einem **dichten LFM2.5-Textmodell**
(Default: `LiquidAI/LFM2.5-1.2B-Instruct`) mit 4-bit-QLoRA.

**Vorher:** `data/train.jsonl` im `messages`-Format nach
`MyDrive/muscal-lfm/text/data/` legen (lokal erzeugen mit
`scripts/prepare_dataset.py`).

**Runtime:** GPU. Eine freie T4 (15 GB) reicht für 350M/1.2B; für 2.6B
`per_device_train_batch_size` auf 1–2 senken.

In [ ]:
# Colab brings torch; we only need the training stack. --upgrade on purpose:
# LFM2.5 checkpoints need a recent transformers.
!pip install -q -U transformers trl peft accelerate bitsandbytes datasets pyyaml

import torch
print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")
    print("bf16 supported:", torch.cuda.is_bf16_supported())

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/muscal-lfm/text')
(DRIVE_DIR / 'data').mkdir(parents=True, exist_ok=True)
(DRIVE_DIR / 'outputs').mkdir(parents=True, exist_ok=True)
print("drive project dir:", DRIVE_DIR)
print("contents:", sorted(p.name for p in DRIVE_DIR.iterdir()))

In [ ]:
# Get the training code. If you already have the repo in Drive, skip this cell.
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/INDIEaner84/MUSCAL-ColabAPI-ProviderLLM.git"
REF = os.environ.get("MUSCAL_REF", "main")   # set to your branch before pushing

if not Path('/content/MUSCAL-ColabAPI-ProviderLLM').exists():
    !git clone --depth 1 --branch $REF $REPO_URL /content/MUSCAL-ColabAPI-ProviderLLM

%cd /content/MUSCAL-ColabAPI-ProviderLLM
sys.path.insert(0, '/content/MUSCAL-ColabAPI-ProviderLLM')
print("cwd:", Path.cwd())

In [ ]:
from muscal_lfm.config import TrainConfig
from muscal_lfm.model import gpu_info
import yaml

CONFIG = "configs/text_qlora.yaml"
cfg = TrainConfig.from_yaml(CONFIG)

# --- your settings -------------------------------------------------------
cfg.data.train_file = str(DRIVE_DIR / "data" / "train.jsonl")
cfg.training.output_dir = str(DRIVE_DIR / "outputs" / "text.yaml")
cfg.training.num_train_epochs = 3
cfg.training.per_device_train_batch_size = 4
cfg.training.gradient_accumulation_steps = 4
cfg.training.learning_rate = 2e-4
# ------------------------------------------------------------------------

print("gpu:", gpu_info())
print()
print(cfg.to_yaml())

## Daten prüfen

Lieber hier scheitern als nach 20 Minuten Training.

In [ ]:
from muscal_lfm import data as data_utils
from datasets import Dataset

ds = data_utils.load_file(cfg.data.train_file)
print("rows:", len(ds), "| columns:", ds.column_names)
print()
print("example row:")
print(ds[0])

problems_exist = False
try:
    data_utils.validate_conversational(ds)
    print("\n[ok] dataset shape looks good")
except data_utils.DatasetError as exc:
    problems_exist = True
    print("\n[problem]", exc)

print("\nstats:", data_utils.dataset_report(ds))

## Trainieren

Der erste Durchlauf lädt das Basis-Modell (~2.5 GB für 1.2B). Danach sind es
bei 500 Beispielen und 3 Epochen auf der T4 typischerweise unter 15 Minuten.

In [ ]:
from muscal_lfm.train import train
from pathlib import Path

Path(cfg.training.output_dir).mkdir(parents=True, exist_ok=True)
Path(cfg.training.output_dir, "config.resolved.yaml").write_text(cfg.to_yaml())

adapter_dir = train(cfg)
print("adapter:", adapter_dir)
print(sorted(p.name for p in Path(adapter_dir).iterdir()))

## Vorher / Nachher

Erst die Basis-Antwort, dann die des feingetunten Modells. Wenn beide gleich
sind: Loss-Kurve prüfen — meistens war die Learning-Rate zu klein oder die Daten
haben nicht das gelernt, was der Probe-Prompt abfragt.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(cfg.model.id)
base = AutoModelForCausalLM.from_pretrained(cfg.model.id, dtype="auto", device_map="auto")
model = PeftModel.from_pretrained(base, str(adapter_dir))

PROBE = "Erklaere in einem Satz, was dieses Modell macht."

messages = [{"role": "user", "content": PROBE}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

print("--- BASE ---")
with torch.no_grad():
    out = base.generate(**inputs, max_new_tokens=64, do_sample=False)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

print("\n--- FINETUNED ---")
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## Adapter zusammenführen

Für den Einsatz in llama.cpp / Ollama / LM Studio wird der Adapter mit dem
Basis-Modell verschmolzen und nach GGUF gewandelt. Der Merge läuft auf CPU, um
den GPU-Speicher nicht zu sprengen.

In [ ]:
from muscal_lfm.export import merge_adapter, export_gguf

merged = merge_adapter(
    cfg.model.id,
    adapter_dir,
    str(Path(cfg.training.output_dir) / "merged"),
    track=cfg.model.track,
)
print("merged:", merged)

# Optional: GGUF for llama.cpp / Ollama / LM Studio.
# Needs a current llama.cpp (LFM2.5 is a young architecture) and ~10 min.
EXPORT_GGUF = False
if EXPORT_GGUF:
    gguf = export_gguf(merged, quant="q4_k_m")
    !cp "$gguf" "$DRIVE_DIR/outputs/"

## Nächste Schritte

* Mehr Daten / andere `r`-Werte: `cfg.lora.r` (16 → 32) erhöhen, dafür LR leicht senken.
* Präferenz-Training: `DPOTrainer` aus TRL, Datensatz mit `prompt`/`chosen`/`rejected`,
  Learning-Rate 1e-7 … 1e-6 statt 2e-4.
* MoE statt dicht: `notebooks/03_lfm_moe_qlora.ipynb`.
* Siehe `docs/gguf-export.md` für den Weg bis aufs Gerät.